In [9]:
from pathlib import Path
import sys

ROOT_DIR = Path().resolve().parents[2]
sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from config.paths import CLEANED_MARKET_FILE

df = pd.read_csv(CLEANED_MARKET_FILE)

In [21]:
df

,date,symbol,open,high,low,close,volume
22291,2001-05-15 10:00:00,EGX:CIEB,0.081230,0.081258,0.081230,0.081258,6.001330e+05
90678,2001-05-16 10:00:00,EGX:ORHD,0.398725,0.398725,0.380764,0.396569,1.058569e+05
22292,2001-05-16 10:00:00,EGX:CIEB,0.081230,0.084031,0.081230,0.082183,7.354396e+05
22293,2001-05-17 10:00:00,EGX:CIEB,0.084031,0.086272,0.084031,0.086132,1.082453e+06
22294,2001-05-20 10:00:00,EGX:CIEB,0.090334,0.090446,0.089633,0.090390,2.371614e+06
...,...,...,...,...,...,...,...
65481,2026-02-19 10:00:00,EGX:FWRY,20.590000,20.890000,19.200000,19.590000,1.301843e+07
114630,2026-02-19 10:00:00,EGX:SWDY,83.490000,84.800000,80.020000,80.750000,1.035574e+06
63892,2026-02-19 10:00:00,EGX:EXPA,17.230000,17.490000,16.800000,17.000000,8.802530e+05
50692,2026-02-19 10:00:00,EGX:EGAL,227.000000,229.400000,225.000000,225.600000,1.330220e+05


In [22]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

In [23]:
recent_3_years = [2023, 2024, 2025]
filtered_df = df[df['date'].dt.year.isin(recent_3_years)]

filtered_df

,date,symbol,open,high,low,close,volume
51010,2023-01-02 00:00:00,EGX:EKHOA,31.437703,32.554778,31.527468,32.514881,519106.00
32518,2023-01-02 00:00:00,EGX:COMI,39.652780,41.000669,39.432912,40.866837,2708615.00
53482,2023-01-02 00:00:00,EGX:ESRS,23.250000,25.520000,23.400000,25.450001,2974334.00
94825,2023-01-02 10:00:00,EGX:ORHD,7.030000,7.320000,7.020000,7.220000,577243.00
80479,2023-01-02 10:00:00,EGX:JUFO,7.832000,7.832000,7.640000,7.672000,412958.75
...,...,...,...,...,...,...,...
109844,2025-12-31 10:00:00,EGX:SUGR,47.410000,47.680000,47.050000,47.190000,38787.00
63859,2025-12-31 10:00:00,EGX:EXPA,15.530000,15.630000,15.340000,15.400000,1087322.00
4966,2025-12-31 10:00:00,EGX:ABUK,50.780000,51.500000,50.700000,51.000000,490093.00
40659,2025-12-31 10:00:00,EGX:EAST,37.300000,37.440000,37.010000,37.100000,530228.00


In [ ]:
monthly_df = df.set_index('date').groupby('symbol')['close'].resample('ME').last().reset_index()
monthly_df['monthly_return'] = monthly_df.groupby('symbol')['close'].pct_change()
monthly_df.dropna(inplace=True)

monthly_df

C:\Users\nasho\AppData\Local\Temp\ipykernel_352\1852066929.py:2: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  monthly_df['monthly_return'] = monthly_df.groupby('symbol')['close'].pct_change()


,symbol,date,close,monthly_return
0,EGX:ABUK,2005-02-28,3.404361,NaN
1,EGX:ABUK,2005-03-31,3.442906,0.011322
2,EGX:ABUK,2005-04-30,3.564360,0.035277
3,EGX:ABUK,2005-05-31,3.364361,-0.056111
4,EGX:ABUK,2005-06-30,3.547997,0.054583
...,...,...,...,...
6552,EGX:TMGH,2025-10-31,57.510000,0.011076
6553,EGX:TMGH,2025-11-30,76.440000,0.329160
6554,EGX:TMGH,2025-12-31,80.000000,0.046572
6555,EGX:TMGH,2026-01-31,86.490000,0.081125


In [25]:
summary = monthly_df.groupby('symbol')['monthly_return'].agg([
    ('avg_annual_return', lambda x: x.mean() * 12),
    ('annualized_sd', lambda x: x.std() * np.sqrt(12))
]).reset_index()

summary

,symbol,avg_annual_return,annualized_sd
0,EGX:ABUK,0.197374,0.339571
1,EGX:ADIB,0.307989,0.539112
2,EGX:AMOC,0.078712,0.371184
3,EGX:BTFH,0.403156,0.802886
4,EGX:CCAP,0.071628,0.517332
5,EGX:CIEB,0.870864,3.407187
6,EGX:CIRA,0.138694,0.453153
7,EGX:COMI,0.298235,0.320661
8,EGX:DOMT,0.028422,0.377221
9,EGX:EAST,0.248876,0.357861


In [26]:
summary['risk_category'] = pd.qcut(
    summary['annualized_sd'], 
    q=3, 
    labels=['Conservative', 'Moderate', 'Aggressive']
)

In [27]:
final_table = summary.sort_values(by=['risk_category', 'avg_annual_return'], ascending=[True, False])
final_table.to_csv('classified_stock_risk.csv', index=False, encoding='utf-8-sig')

print("Process Complete. CSV saved with Annualized metrics.")
print(final_table)

Process Complete. CSV saved with Annualized metrics.
       symbol  avg_annual_return  annualized_sd risk_category
7    EGX:COMI           0.298235       0.320661  Conservative
9    EGX:EAST           0.248876       0.357861  Conservative
0    EGX:ABUK           0.197374       0.339571  Conservative
15   EGX:EXPA           0.192076       0.402375  Conservative
25   EGX:ORWE           0.142312       0.379322  Conservative
14   EGX:ETEL           0.124859       0.308648  Conservative
23   EGX:MTIE           0.121354       0.342801  Conservative
12  EGX:EKHOA           0.111389       0.304016  Conservative
2    EGX:AMOC           0.078712       0.371184  Conservative
8    EGX:DOMT           0.028422       0.377221  Conservative
16   EGX:FWRY           0.427635       0.443187      Moderate
22   EGX:MFPC           0.394408       0.478790      Moderate
28   EGX:SWDY           0.268022       0.417297      Moderate
10   EGX:EFIC           0.257971       0.486710      Moderate
20   EGX:JUFO    